In [314]:
import math
import os
from collections.abc import Callable
from math import pow, sqrt

import forallpeople as units
import handcalcs
import handcalcs.render
import numpy as np

# import polars as pl
import pandas as pd
import sympy as sp
from IPython.display import Latex, display

from structural_tools.seismic.parameters import (
    RiskCategory,
    SeismicParameters,
    SiteClass,
)

units.environment(env_name="structural", top_level=True)
handcalcs.set_option("custom_symbols", {"star": "^*", "__": ","})
handcalcs.set_option("greek_exclusions", ["psi"])
handcalcs.set_option("display_precision", 3)
handcalcs.set_option("param_columns", 1)
handcalcs.set_option("latex_block_start", "\\begin{equation*}")
handcalcs.set_option("latex_block_end", "\\end{equation*}")
# handcalcs.set_option("math_environment_start", "aligned")
# handcalcs.set_option("math_environment_end", "aligned")


def check_value(value: float, check_value: float = 1, inequality: str = "leq"):
    pass_latex = r"\textbf{\color{OK}OK}\ {\color{OK}\checkmark}"
    fail_latex = r"\textbf{{\color{NG}NG !}}"
    outcome = fail_latex
    operator = "ERROR"
    match inequality:
        case "leq":
            if value <= check_value:
                outcome = pass_latex
                operator = "\\leq"
            else:
                operator = "\\gt"
        case "lt":
            if value < check_value:
                outcome = pass_latex
                operator = "\\lt"
            else:
                operator = "\\geq"
        case "geq":
            if value >= check_value:
                outcome = pass_latex
                operator = "\\geq"
            else:
                operator = "\\lt"
        case "gt":
            if value > check_value:
                outcome = pass_latex
                operator = "\\gt"
            else:
                operator = "\\leq"
    return f"{value:.3g} {operator} {check_value:.3g} \\quad {outcome}"


def display_table(
    styler: pd.io.formats.style.Styler | None = None,
    position_float="centering",
    hrules=True,
    position="H",
    **kwargs,
) -> Latex | None:
    """Displays a table if run in a Jupyter ipynb, or returns LaTeX code if run by nbconvert when exporting to PDF

    Args:
        styler (pd.io.formats.style.Styler): Formatted table styler created by the `create_formatted_table` function
        position_float (str, optional): See pandas documentation for Styler.to_latex(). Defaults to "centering".
        hrules (bool, optional): See pandas documentation for Styler.to_latex(). Defaults to True.
        position (str, optional): See pandas documentation for Styler.to_latex(). Defaults to "H".

    Returns:
        _type_: _description_
    """
    # Need to use the "export_to_*" scripts to set the NBCONVERT environment variable for this to work,
    # otherwise it will just display the dataframe as normal without LaTeX formatting
    if "NBCONVERT" in os.environ:
        return Latex(
            styler.to_latex(
                position_float=position_float,
                hrules=hrules,
                position=position,
                **kwargs,
            )
        )
    else:
        display(styler.data)


def sig_figs(x: float, sig_figs: int):
    """
    Rounds a number to number of significant figures
    Parameters:
    - x - the number to be rounded
    - precision (integer) - the number of significant figures
    Returns:
    - float
    """

    if pd.isna(x):
        return ""

    if x == 0:
        return float(0)

    x = float(x)
    sig_figs = int(sig_figs)

    if 1e-3 <= abs(x) < 1e6:
        decimals = sig_figs - int(math.floor(math.log10(abs(x)))) - 1
        decimals = max(decimals, 0)
        return f"{x:.{decimals}f}"

    return round(x, -int(math.floor(math.log10(abs(x)))) + (sig_figs - 1))

# Effective Seismic Weight Calculation

## Unit Weights

In [315]:
%%render params
U_corridor = 35 / 1000 * ksf
U_floor = 30 / 1000 * ksf
U_roof = 25 / 1000 * ksf
U_wall = 12 / 1000 * ksf

<IPython.core.display.Latex object>

## Building Geometry

In [316]:
%%render param
L_floor = 235 * ft
w_floor = (30 + 9 / 12) * ft
L_corridor = 235 * ft
w_corridor = 5.5 * ft
w_building = (w_floor * 2) + w_corridor
h_wall__1 = 11.5 * ft
h_wall__2 = 10.5 * ft
h_wall__3 = 10.5 * ft
h_wall__4 = 10 * ft
L_jog = 5 * ft
P_wall = (2 * L_floor) + (2 * w_building) + (2 * L_jog)

<IPython.core.display.Latex object>

In [317]:
%%render 
A_floor = 2 * (L_floor * w_floor)
# This is a slight overestimate as the corridor doesn't stretch the full length of the building
A_corridor = L_corridor * w_corridor
A_level = L_floor * w_building
A_wall__1 = h_wall__1 * P_wall
A_wall__2 = h_wall__2 * P_wall
A_wall__3 = h_wall__3 * P_wall
A_wall__4 = h_wall__4 * P_wall

<IPython.core.display.Latex object>

## Component Weights

In [318]:
%%render long
W_floor__2 = U_floor * A_floor
W_floor__3 = W_floor__2
W_floor__4 = W_floor__2
W_corridor__2 = U_corridor * A_corridor
W_corridor__3 = W_corridor__2
W_corridor__4 = W_corridor__2
W_roof = U_roof * A_level
# W_truss = U_truss * A_level
W_wall__2 = U_wall * (A_wall__2 / 2 + A_wall__1 / 2)
W_wall__3 = U_wall * (A_wall__3 / 2 + A_wall__2 / 2)
W_wall__4 = U_wall * (A_wall__4 / 2 + A_wall__3 / 2)
W_wall__roof = U_wall * (A_wall__4 / 2)

<IPython.core.display.Latex object>

## Level Weights

In [319]:
%%render long
W_level__2 = W_floor__2 + W_corridor__2 + W_wall__2
W_level__3 = W_floor__3 + W_corridor__3 + W_wall__3
W_level__4 = W_floor__4 + W_corridor__4 + W_wall__4
# be sure to add the weight of the truss later
W_level__roof = W_roof + W_wall__roof  # + Wtruss
W_total = W_level__2 + W_level__3 + W_level__4 + W_level__roof

<IPython.core.display.Latex object>

# Seismic Equivalent Lateral Force (ELF) Analysis

We will be using the equivalent lateral force method prescribed by ASCE 7-16 Section 12.8. We will envelope the forces with a fully flexible diaphragm assumption and a fully rigid diaphragm assumption.

## Seismic Parameters

In [320]:
S_S = 0.887  # g
S_1 = 0.410  # g
seismic_params = SeismicParameters(
    s_1=S_1, s_s=S_S, risk_category=RiskCategory.II, site_class=SiteClass.D
)

In [321]:
%%render params 3
R = 6.5
Omega_0 = 3.0
C_d = 4.0
rho = 1.0
I_e = seismic_params.i_e
S_S = seismic_params.s_s  # g
S_1 = seismic_params.s_1  # g
F_a = seismic_params.f_a
F_v = seismic_params.f_v
S_MS = seismic_params.s_ms  # g
S_M1 = seismic_params.s_m1  # g
S_DS = seismic_params.s_ds  # g
S_D1 = seismic_params.s_d1  # g
SDC = seismic_params.sdc
C_u = 1.4
T_L = 16  # s

<IPython.core.display.Latex object>

## Base Shear

We must calculate the period

In [322]:
%%render params
h_top__roof = (57 + 8 / 12) * ft
h_bottom__roof = 45.5 * ft
h_n = np.mean([h_top__roof, h_bottom__roof])  # ASCE 7-16 Sec. 11.2

<IPython.core.display.Latex object>

In [323]:
%%render
h_n__check = check_value(h_n, 65 * ft, "leq")

<IPython.core.display.Latex object>

Building A is fewer than 12 stories, and the average story height is greater than 10 ft, so

In [324]:
%%render
N_stories = 4
T_a = 0.1 * N_stories  # ASCE 7-16 Eq. 12.8-8

<IPython.core.display.Latex object>

Because we are not using an automated analysis software, we can make the following assumption:

In [325]:
%%render
T = T_a  # ASCE 7-16 Sec. 12.8.2

<IPython.core.display.Latex object>

With the period, we can calculate the seismic response coefficient.

In [326]:
%%render
C_s = S_DS / (R / I_e)  # ASCE 7-16 Eq. 12.8-2

<IPython.core.display.Latex object>

We must check that $C_s$ does not exceed:

In [327]:
%%render long
# fmt: off
if T <= T_L:  C_s__check__max = S_D1 / (T * (R / I_e));              C_s__check__max = check_value(C_s, C_s__check__max, "leq")    # ASCE 7-16 Sec. 12.8.1.1
elif T > T_L: C_s__check__max = (S_D1 * T_L) / (T**2 * (R / I_e));   C_s__check__max = check_value(C_s, C_s__check__max, "leq")  # ASCE 7-16 Sec. 12.8.1.1
# fmt: on

<IPython.core.display.Latex object>

And we must also check that $C_s$ is not less than:

In [328]:
%%render
C_s__check__min__1 = 0.044 * S_DS * I_e  # ASCE 7-16 Eq. 12.8-5
C_s__check__min = check_value(C_s, C_s__check__min__1, "geq")

<IPython.core.display.Latex object>

as well as:

In [329]:
%%render
C_s__check__min = check_value(C_s, 0.01, "geq")  # ASCE 7-16 Eq. 12.8-5

<IPython.core.display.Latex object>

and finally:

In [330]:
%%render
# fmt: off
if S_1 >= 0.6:  C_s__check__min__3 = 0.5 * S_1 / (R / I_e); C_s__check__min = check_value(C_s, C_s__check__min__3, "geq")  # ASCE 7-16 Eq. 12.8-6
elif S_1 < 0.6: C_s__check__min = "\\text{skipping check}"  # ASCE 7-16 Eq. 12.8-6
# fmt: on

<IPython.core.display.Latex object>

And now, we can calculate the base shear, $V$

In [331]:
%%render
V = C_s * W_total  # ASCE 7-16 Eq. 12.8-1

<IPython.core.display.Latex object>

## Distribute the seismic forces

With the base shear calculated, we must now distribute the base shear on each level.

First, we must calculate the structural period exponent, $k$.

In [332]:
%%render
# fmt: off
if T <= 0.5: k = 1
elif T >= 2.5: k = 2
elif T > 0.5 and T < 2.5: k = 1 + ((T - 0.5) * (2 - 1)) / (2.5 - 0.5)
# fmt: on

<IPython.core.display.Latex object>

Now, we can distribute the forces per floor,

$$
F_x = C_{vx} V\;\;\text{(ASCE 7-16 Eq. 12.8-11)}
$$

As well as cumulatively from roof to ground.

$$
V_x = \sum_{i=x}^n F_i\;\; \text{(ASCE 7-16 Eq. 12.8-11)}
$$

In [333]:
# Load in basic data
df = pd.DataFrame(
    {
        "level": ["ground", "level 2", "level 3", "level 4", "roof"],
        "floor_height": [
            float(h_wall__1),
            float(h_wall__2),
            float(h_wall__3),
            float(h_wall__4),
            0,
        ],
        "floor_weight": [
            0,
            float(W_level__2),
            float(W_level__3),
            float(W_level__4),
            float(W_level__roof),
        ],
    },
)

# Calc cumulative elevation
elevations = [0]
for height in df.floor_height[:-1]:
    elevations.append(height + elevations[-1])
df.insert(2, "elevation", elevations)

# Calc coefficients and story forces ASCE 7-16 Eq. 12.8-11 and 12.8-12
df["wxhxk"] = df.floor_weight * df.elevation ** float(k)
cumulative_wxhxk = df.wxhxk.sum()
df["C_vx"] = df.wxhxk / cumulative_wxhxk
df["F_x"] = df.C_vx * float(V)

# Calc story shear forces ASCE 7-16 Eq. 12.8-13
story_shears = [0]
for shear in reversed(list(df.F_x[1:])):
    story_shears.insert(0, shear + story_shears[0])
df["V_x"] = story_shears

# Calc overturning moments
floor_moments = df.V_x * df.floor_height
overturning_moments = [0]
for floor_moment in reversed(list(floor_moments[:-1])):
    overturning_moments.insert(0, overturning_moments[0] + floor_moment)
df["OTM"] = overturning_moments

In [334]:
# Rename columns for display with LaTeX formatting for units and subscripts/superscripts
column_name_map = {
    "level": "Level",
    "floor_height": "$h_{floor}$ [ft]",
    "floor_weight": "$w_x$ [kip]",
    "elevation": "$h_x$ [ft]",
    "wxhxk": "$w_x h_x^k$",
    "C_vx": "$C_{vx}$",
    "F_x": "$F_x$ [kip]",
    "V_x": "$V_x$ [kip]",
    "OTM": "OTM [kip-ft]",
}
df.rename(columns=column_name_map, errors="raise", inplace=True)


# Format the dataframe
# Reverse the order of the dataframe with '.loc[::-1]' so roof is the top row and ground is the bottom row, then apply formatting with the 'format_dataframe' function
styler = df.iloc[::-1].style
styler = styler.format(lambda value: sig_figs(value, sig_figs=3), subset=df.columns[1:])
styler = styler.hide(axis="index")

Calculating for each floor leads to the following \autoref{tab:seismic_forces_calculation_table}

In [335]:
display_table(
    styler=styler,
    caption="Seismic Forces Calculation Table",
    label="tab:seismic_forces_calculation_table",
)

,Level,$h_{floor}$ [ft],$h_x$ [ft],$w_x$ [kip],$w_x h_x^k$,$C_{vx}$,$F_x$ [kip],$V_x$ [kip],OTM [kip-ft]
4,roof,0.0,42.5,430.4650,18294.76250,0.332723,72.824219,0.000000,0.000000
3,level 4,10.0,32.5,554.3345,18015.87125,0.327651,71.714063,72.824219,728.242189
2,level 3,10.5,22.0,556.1765,12235.88300,0.222532,48.706214,144.538282,2245.894151
1,level 2,10.5,11.5,559.8605,6438.39575,0.117094,25.628709,193.244496,4274.961362
0,ground,11.5,0.0,0.0000,0.00000,0.000000,0.000000,218.873205,6792.003221


#### 